# Membrane mechanics with mem3dg

_Investigation `membrane-mechanics-with-mem3dg` — coder reproduction notebook._

**Question.** TBD — what overarching question ties these studies together?

TBD — one paragraph, for a reader who has never seen this.

---

**How to use this notebook**

1. **Run ▸ Run All Cells** to reproduce every study end to end.
2. To explore, edit a study's **Parameters** cell, then re-run from its **Run** section — the figures regenerate with your values.
3. The **Setup** cell at the top is collapsed scaffolding (environment + render helpers); expand it only to inspect. Set `RERUN = False` there to render the committed `runs.db` without re-simulating.

The prose states the **question and parameters**; the figures each run produces are the results.


In [ ]:
"""Setup — environment & helpers (runs automatically; open only to inspect).

Generated by vivarium-workbench (notebook_export). This cell resolves the repo,
registers the workspace Core, and defines the render helpers; the science lives
in the sections below. Each study is re-run live with the workspace's own
process-bigraph protocol and its figures rendered from the resulting runs.db.
"""
import os
import sys
from pathlib import Path

# Resolve the repository root robustly so this notebook runs from a fresh clone
# at ANY path with no setup (no env var, no path editing). Priority:
#   1. $VIVARIUM_REPO, if it points at a real directory;
#   2. walk up from the notebook's working directory for the repo markers
#      (a directory holding both 'workspace/' and 'pyproject.toml') — Jupyter
#      starts in the notebook's dir, so a committed notebook finds its own root;
#   3. the absolute path it was generated for (back-compat for old layouts);
#   4. the current working directory (last resort).
def _find_repo_root(_start):
    for _cand in (_start, *_start.parents):
        if (_cand / "workspace").is_dir() and (_cand / "pyproject.toml").is_file():
            return _cand
    return None

REPO = None
_env = os.environ.get("VIVARIUM_REPO")
if _env and Path(_env).is_dir():
    REPO = Path(_env)
if REPO is None:
    REPO = _find_repo_root(Path.cwd().resolve())
if REPO is None and Path('/home/runner/work/viva-mem3dg/viva-mem3dg').is_dir():
    REPO = Path('/home/runner/work/viva-mem3dg/viva-mem3dg')
if REPO is None:
    REPO = Path.cwd()
sys.path.insert(0, str(REPO))
# Composite specs use repo-root-relative paths (datasets, caches), and the
# workspace's runner/renderer assume cwd == repo root — so run from there.
os.chdir(REPO)

# Re-simulate from scratch? Set False to render the committed runs.db (fast).
RERUN = True

# Keep the output clean: composite discovery and the pydantic models emit
# benign UserWarnings (optional packages it skips, a field-name shadow) and the
# simulators emit DeprecationWarnings — none are actionable from a reproduction
# notebook, so silence those two categories. Remove this to see them.
import warnings as _warnings
_warnings.filterwarnings("ignore", category=UserWarning)
_warnings.filterwarnings("ignore", category=DeprecationWarning)

# --- standard process-bigraph protocol: register the workspace's Core ---
from pbg_mem3dg.core import build_core
core = build_core()

# --- imported from the repo this notebook was generated for ---

from IPython.display import HTML, display

import contextlib as _contextlib, io as _io
@_contextlib.contextmanager
def quiet():
    """Silence the simulator's verbose per-step stdout so the notebook
    output stays readable (the figures below are the results)."""
    with _contextlib.redirect_stdout(_io.StringIO()):
        yield

import html as _htmlmod
def show_viz(_h, height=560):
    """Display a visualization's HTML in an isolated iframe.

    The figures embed their own scripts (e.g. Plotly); JupyterLab does not
    execute <script> tags from display(HTML(...)), so an iframe srcdoc is
    used instead — the browser runs the scripts inside the frame."""
    display(HTML(
        '<iframe srcdoc="{}" style="width:100%;height:{}px;border:0">'
        '</iframe>'.format(_htmlmod.escape(_h, quote=True), height)
    ))

import json as _json
def describe_spec(spec):
    """Print a composite spec's structure (parameters, processes, wiring)
    then the full editable dict. The spec is plain data — assign to any
    field (e.g. spec['state'][proc]['config'][...]) before building."""
    print("composite:", spec.get("name"))
    if spec.get("description"):
        print("description:", str(spec["description"]).strip())
    _params = spec.get("parameters") or {}
    if _params:
        print("\nparameters (filled into ${name} placeholders):")
        for _p, _pdef in _params.items():
            print(f"  {_p}: default={_pdef.get('default')!r}  type={_pdef.get('type')}")
    print("\nprocesses (node -> address):")
    for _node, _body in (spec.get("state") or {}).items():
        if not (isinstance(_body, dict) and _body.get("_type") == "process"):
            continue
        print(f"  {_node}  ->  {_body.get('address')}   interval={_body.get('interval')!r}")
        for _port in ("inputs", "outputs"):
            if _body.get(_port):
                print(f"      {_port} ports: {_body[_port]}")
    print("\nfull editable spec dict:")
    print(_json.dumps(spec, indent=2, default=str))

def materialize_composite(spec_id, overrides=None):
    """Build a composite into an editable payload {name, description,
    parameters, state, _overrides, _id}. Works for @composite_generator
    functions via the process-bigraph registry (the common case); ``state``
    is the built document (process nodes at top level)."""
    import importlib
    from process_bigraph.composite_generator import (
        _REGISTRY, apply_core_extensions, build_generator)
    try:
        from process_bigraph.composite_generator import discover_generators
        discover_generators()
    except Exception:
        pass
    entry = _REGISTRY.get(spec_id)
    if entry is None:
        # editable installs are invisible to packages_distributions, so
        # discover alone can miss them — prime the module the id names.
        try:
            importlib.import_module(spec_id.rsplit('.', 1)[0])
            from process_bigraph.composite_generator import discover_generators as _dg
            _dg()
        except Exception:
            pass
        entry = _REGISTRY.get(spec_id)
    if entry is None:
        raise KeyError('composite not registered: ' + repr(spec_id))
    params = dict(entry.parameters or {})
    ov = {k: (v.get('default') if isinstance(v, dict) else None)
          for k, v in params.items()}
    if overrides:
        ov.update({k: val for k, val in overrides.items() if k in params})
    _c = apply_core_extensions(entry, core)
    try:
        doc = build_generator(entry, ov or None, core=_c)
    except TypeError:
        doc = entry.func(_c, **(ov or {}))
    return {'name': entry.name, 'description': entry.description,
            'parameters': params, 'state': doc, '_overrides': ov, '_id': spec_id}

import base64 as _b64, pathlib as _pl
_STATIC_VIZ_SCHEMES = ('image:', 'file:', 'gif:', 'png:', 'svg:', 'jpg:', 'jpeg:')
def _render_viz_process(address, config):
    """Render a Visualization process/step (e.g. ``local:<Class>``) to
    HTML: wrap it as a one-step composite and run it once — the viz writes
    Plotly HTML to its ``html`` output port. Mirrors the workbench's own
    env-worker viz path. Self-running vizes need no run data."""
    from process_bigraph import Composite
    _doc = {
        'inputs_store': {}, 'output_store': '',
        'visualization': {
            '_type': 'step', 'address': address,
            'config': {k: v for k, v in (config or {}).items()
                       if k not in ('inputs_map', 'sources')},
            'inputs': {}, 'outputs': {'html': ['output_store']},
        },
    }
    _comp = Composite({'state': _doc}, core=core)
    _comp.run(1)
    _html = _comp.state.get('output_store')
    if isinstance(_html, dict):
        _html = _html.get('value') or _html.get('_value') or ''
    return _html if isinstance(_html, str) else ''
def _render_one(address, config, runs_db, study_yaml, overrides=None):
    """Generic figure renderer (no workspace render_study_viz.py).
    A static ``image:``/``file:`` (or bare-path) address embeds the file;
    any other address (``local:<VizProcess>``) is run as a Visualization
    and its HTML returned. ``overrides`` (the composite's edited parameter
    dict) is merged into the viz config first so a self-running figure
    reflects edited parameters; a viz that does not accept those keys
    falls back to the base config."""
    addr = str(address or '')
    _head = addr.split('/', 1)[0]
    _is_static = (not addr) or (':' not in _head) or addr.startswith(_STATIC_VIZ_SCHEMES)
    if not _is_static:
        if core is None:
            return f'<p style="color:#b91c1c">no core to render {address}</p>'
        _base = {k: v for k, v in (config or {}).items()}
        _tries = ([{**_base, **overrides}] if overrides else []) + [_base]
        _err = ''
        for _cfg in _tries:
            try:
                _h = _render_viz_process(addr, _cfg)
            except Exception as _e:
                _err = (f'<p style="color:#b91c1c">visualization {address} not renderable yet: '
                        f'{type(_e).__name__}: {_e}</p>')
                continue
            return _h or f'<p style="color:#6b7280">{address} produced no HTML</p>'
        return _err
    for _scheme in _STATIC_VIZ_SCHEMES:
        if addr.startswith(_scheme):
            addr = addr[len(_scheme):]; break
    _p = _pl.Path(addr)
    if not _p.is_absolute():
        _p = _pl.Path(study_yaml).resolve().parent / _p
    if not _p.is_file():
        return f'<p style="color:#b91c1c">figure not found: {address}</p>'
    _suffix = _p.suffix.lower()
    if _suffix == '.svg':
        return _p.read_text(encoding='utf-8', errors='replace')
    if _suffix in ('.png', '.jpg', '.jpeg', '.gif', '.webp'):
        _mime = 'jpeg' if _suffix in ('.jpg', '.jpeg') else _suffix[1:]
        _data = _b64.b64encode(_p.read_bytes()).decode('ascii')
        return f'<img src="data:image/{_mime};base64,{_data}" style="max-width:100%"/>'
    if _suffix in ('.html', '.htm'):
        return _p.read_text(encoding='utf-8', errors='replace')
    return f'<p style="color:#6b7280">unsupported figure type: {address}</p>'

## Study: Membrane demo (`membrane-demo`)

**Question.** TBD — what question does this study answer?


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `pbg_mem3dg.composites.membrane-demo` | 30 | interval=1.0 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**Composite `pbg_mem3dg.composites.membrane-demo`** — `spec_pbg_mem3dg_composites_membrane_demo` (static spec file, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_pbg_mem3dg_composites_membrane_demo = load_spec(REPO / 'pbg_mem3dg/composites/membrane-demo.composite.yaml')
describe_spec(spec_pbg_mem3dg_composites_membrane_demo)

In [ ]:
# === Edit parameters for composite 'membrane-demo' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# tunable parameters (filled into ${name} placeholders):
spec_pbg_mem3dg_composites_membrane_demo['parameters']['interval']['default'] = 5.0
spec_pbg_mem3dg_composites_membrane_demo['parameters']['subdivision']['default'] = 2
spec_pbg_mem3dg_composites_membrane_demo['parameters']['characteristic_timestep']['default'] = 1.0
spec_pbg_mem3dg_composites_membrane_demo['parameters']['Kbc']['default'] = 8.22e-05
spec_pbg_mem3dg_composites_membrane_demo['parameters']['tension_modulus']['default'] = 0.1
spec_pbg_mem3dg_composites_membrane_demo['parameters']['osmotic_strength']['default'] = 0.02
spec_pbg_mem3dg_composites_membrane_demo['parameters']['preferred_volume_fraction']['default'] = 0.7
spec_pbg_mem3dg_composites_membrane_demo['parameters']['viz_title']['default'] = 'Mem3DG membrane relaxation'
spec_pbg_mem3dg_composites_membrane_demo['parameters']['viz_3d_title']['default'] = 'Membrane mesh'

# process 'membrane'  (local:Mem3DGProcess)
# spec_pbg_mem3dg_composites_membrane_demo['state']['membrane']['interval'] = 0.01   # pin this process's dt (else filled by INTERVAL below)
spec_pbg_mem3dg_composites_membrane_demo['state']['membrane']['config']['mesh_type'] = 'icosphere'
spec_pbg_mem3dg_composites_membrane_demo['state']['membrane']['config']['radius'] = 1.0
spec_pbg_mem3dg_composites_membrane_demo['state']['membrane']['config']['subdivision'] = '${subdivision}'
spec_pbg_mem3dg_composites_membrane_demo['state']['membrane']['config']['Kbc'] = '${Kbc}'
spec_pbg_mem3dg_composites_membrane_demo['state']['membrane']['config']['tension_modulus'] = '${tension_modulus}'
spec_pbg_mem3dg_composites_membrane_demo['state']['membrane']['config']['osmotic_strength'] = '${osmotic_strength}'
spec_pbg_mem3dg_composites_membrane_demo['state']['membrane']['config']['preferred_volume_fraction'] = '${preferred_volume_fraction}'
spec_pbg_mem3dg_composites_membrane_demo['state']['membrane']['config']['characteristic_timestep'] = '${characteristic_timestep}'

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: membrane-demo ===
STUDY = 'membrane-demo'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# Reproduce the recorded run(s). STEPS = composite steps.
STEPS_baseline = 30
INTERVAL_baseline = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_mem3dg_composites_membrane_demo, {'interval': INTERVAL_baseline}, core=core)
            _comp.run(STEPS_baseline)
        print('ran', 'baseline', f'(pbg_mem3dg.composites.membrane-demo) for', STEPS_baseline, 'steps')
    except Exception as _e:
        print(f"baseline not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**membrane-demo-world**


In [ ]:
# membrane-demo-world
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**membrane-demo-mechanism**


In [ ]:
# membrane-demo-mechanism
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**membrane-demo-decision**


In [ ]:
# membrane-demo-decision
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| membrane-demo-behaves _(primary)_ | kind=derived_scalar field=TBD | op < value None |


## Study: Membrane patch (`membrane-patch`)

**Question.** TBD — what question does this study answer?


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `pbg_mem3dg.composites.membrane-patch` | 30 | interval=1.0 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**Composite `pbg_mem3dg.composites.membrane-patch`** — `spec_pbg_mem3dg_composites_membrane_patch` (static spec file, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_pbg_mem3dg_composites_membrane_patch = load_spec(REPO / 'pbg_mem3dg/composites/membrane-patch.composite.yaml')
describe_spec(spec_pbg_mem3dg_composites_membrane_patch)

In [ ]:
# === Edit parameters for composite 'membrane-patch' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# tunable parameters (filled into ${name} placeholders):
spec_pbg_mem3dg_composites_membrane_patch['parameters']['interval']['default'] = 5.0
spec_pbg_mem3dg_composites_membrane_patch['parameters']['mesh_type']['default'] = 'hexagon'
spec_pbg_mem3dg_composites_membrane_patch['parameters']['radius']['default'] = 1.0
spec_pbg_mem3dg_composites_membrane_patch['parameters']['subdivision']['default'] = 4
spec_pbg_mem3dg_composites_membrane_patch['parameters']['Kbc']['default'] = 0.0003
spec_pbg_mem3dg_composites_membrane_patch['parameters']['tension_modulus']['default'] = 0.01
spec_pbg_mem3dg_composites_membrane_patch['parameters']['preferred_area_scale']['default'] = 1.2
spec_pbg_mem3dg_composites_membrane_patch['parameters']['osmotic_model']['default'] = 'constant'
spec_pbg_mem3dg_composites_membrane_patch['parameters']['osmotic_pressure']['default'] = 0.08
spec_pbg_mem3dg_composites_membrane_patch['parameters']['boundary_condition']['default'] = 'fixed'
spec_pbg_mem3dg_composites_membrane_patch['parameters']['characteristic_timestep']['default'] = 0.5
spec_pbg_mem3dg_composites_membrane_patch['parameters']['tolerance']['default'] = 1e-14
spec_pbg_mem3dg_composites_membrane_patch['parameters']['viz_title']['default'] = 'Patch — energy + geometry'
spec_pbg_mem3dg_composites_membrane_patch['parameters']['viz_3d_title']['default'] = 'Patch mesh'

# process 'membrane'  (local:Mem3DGProcess)
# spec_pbg_mem3dg_composites_membrane_patch['state']['membrane']['interval'] = 0.01   # pin this process's dt (else filled by INTERVAL below)
spec_pbg_mem3dg_composites_membrane_patch['state']['membrane']['config']['mesh_type'] = '${mesh_type}'
spec_pbg_mem3dg_composites_membrane_patch['state']['membrane']['config']['radius'] = '${radius}'
spec_pbg_mem3dg_composites_membrane_patch['state']['membrane']['config']['subdivision'] = '${subdivision}'
spec_pbg_mem3dg_composites_membrane_patch['state']['membrane']['config']['Kbc'] = '${Kbc}'
spec_pbg_mem3dg_composites_membrane_patch['state']['membrane']['config']['tension_modulus'] = '${tension_modulus}'
spec_pbg_mem3dg_composites_membrane_patch['state']['membrane']['config']['preferred_area_scale'] = '${preferred_area_scale}'
spec_pbg_mem3dg_composites_membrane_patch['state']['membrane']['config']['osmotic_model'] = '${osmotic_model}'
spec_pbg_mem3dg_composites_membrane_patch['state']['membrane']['config']['osmotic_pressure'] = '${osmotic_pressure}'
spec_pbg_mem3dg_composites_membrane_patch['state']['membrane']['config']['boundary_condition'] = '${boundary_condition}'
spec_pbg_mem3dg_composites_membrane_patch['state']['membrane']['config']['characteristic_timestep'] = '${characteristic_timestep}'
spec_pbg_mem3dg_composites_membrane_patch['state']['membrane']['config']['tolerance'] = '${tolerance}'

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: membrane-patch ===
STUDY = 'membrane-patch'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# Reproduce the recorded run(s). STEPS = composite steps.
STEPS_baseline = 30
INTERVAL_baseline = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_mem3dg_composites_membrane_patch, {'interval': INTERVAL_baseline}, core=core)
            _comp.run(STEPS_baseline)
        print('ran', 'baseline', f'(pbg_mem3dg.composites.membrane-patch) for', STEPS_baseline, 'steps')
    except Exception as _e:
        print(f"baseline not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**membrane-patch-world**


In [ ]:
# membrane-patch-world
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**membrane-patch-mechanism**


In [ ]:
# membrane-patch-mechanism
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**membrane-patch-decision**


In [ ]:
# membrane-patch-decision
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| membrane-patch-behaves _(primary)_ | kind=derived_scalar field=TBD | op < value None |


## Study: Membrane deflation (`membrane-deflation`)

**Question.** TBD — what question does this study answer?


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `pbg_mem3dg.composites.membrane-deflation` | 30 | interval=1.0 |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**Composite `pbg_mem3dg.composites.membrane-deflation`** — `spec_pbg_mem3dg_composites_membrane_deflation` (static spec file, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_pbg_mem3dg_composites_membrane_deflation = load_spec(REPO / 'pbg_mem3dg/composites/membrane-deflation.composite.yaml')
describe_spec(spec_pbg_mem3dg_composites_membrane_deflation)

In [ ]:
# === Edit parameters for composite 'membrane-deflation' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# tunable parameters (filled into ${name} placeholders):
spec_pbg_mem3dg_composites_membrane_deflation['parameters']['interval']['default'] = 5.0
spec_pbg_mem3dg_composites_membrane_deflation['parameters']['mesh_type']['default'] = 'icosphere'
spec_pbg_mem3dg_composites_membrane_deflation['parameters']['radius']['default'] = 1.0
spec_pbg_mem3dg_composites_membrane_deflation['parameters']['subdivision']['default'] = 3
spec_pbg_mem3dg_composites_membrane_deflation['parameters']['Kbc']['default'] = 0.0005
spec_pbg_mem3dg_composites_membrane_deflation['parameters']['tension_modulus']['default'] = 0.01
spec_pbg_mem3dg_composites_membrane_deflation['parameters']['osmotic_strength']['default'] = 0.1
spec_pbg_mem3dg_composites_membrane_deflation['parameters']['preferred_volume_fraction']['default'] = 0.5
spec_pbg_mem3dg_composites_membrane_deflation['parameters']['characteristic_timestep']['default'] = 1.0
spec_pbg_mem3dg_composites_membrane_deflation['parameters']['tolerance']['default'] = 1e-12
spec_pbg_mem3dg_composites_membrane_deflation['parameters']['viz_title']['default'] = 'Deflation — energy + geometry'
spec_pbg_mem3dg_composites_membrane_deflation['parameters']['viz_3d_title']['default'] = 'Deflation mesh'

# process 'membrane'  (local:Mem3DGProcess)
# spec_pbg_mem3dg_composites_membrane_deflation['state']['membrane']['interval'] = 0.01   # pin this process's dt (else filled by INTERVAL below)
spec_pbg_mem3dg_composites_membrane_deflation['state']['membrane']['config']['mesh_type'] = '${mesh_type}'
spec_pbg_mem3dg_composites_membrane_deflation['state']['membrane']['config']['radius'] = '${radius}'
spec_pbg_mem3dg_composites_membrane_deflation['state']['membrane']['config']['subdivision'] = '${subdivision}'
spec_pbg_mem3dg_composites_membrane_deflation['state']['membrane']['config']['Kbc'] = '${Kbc}'
spec_pbg_mem3dg_composites_membrane_deflation['state']['membrane']['config']['tension_modulus'] = '${tension_modulus}'
spec_pbg_mem3dg_composites_membrane_deflation['state']['membrane']['config']['osmotic_strength'] = '${osmotic_strength}'
spec_pbg_mem3dg_composites_membrane_deflation['state']['membrane']['config']['preferred_volume_fraction'] = '${preferred_volume_fraction}'
spec_pbg_mem3dg_composites_membrane_deflation['state']['membrane']['config']['characteristic_timestep'] = '${characteristic_timestep}'
spec_pbg_mem3dg_composites_membrane_deflation['state']['membrane']['config']['tolerance'] = '${tolerance}'

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: membrane-deflation ===
STUDY = 'membrane-deflation'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# Reproduce the recorded run(s). STEPS = composite steps.
STEPS_baseline = 30
INTERVAL_baseline = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            from viva_superpowers.composite_spec import build_composite_from_spec
            _comp = build_composite_from_spec(spec_pbg_mem3dg_composites_membrane_deflation, {'interval': INTERVAL_baseline}, core=core)
            _comp.run(STEPS_baseline)
        print('ran', 'baseline', f'(pbg_mem3dg.composites.membrane-deflation) for', STEPS_baseline, 'steps')
    except Exception as _e:
        print(f"baseline not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**membrane-deflation-world**


In [ ]:
# membrane-deflation-world
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**membrane-deflation-mechanism**


In [ ]:
# membrane-deflation-mechanism
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

**membrane-deflation-decision**


In [ ]:
# membrane-deflation-decision
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| membrane-deflation-behaves _(primary)_ | kind=derived_scalar field=TBD | op < value None |
